# PySpark drills 3: types, functions and nulls

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-drills-03-types-functions-and-nulls.ipynb)

Practice for [PySpark 3: types, functions and nulls](pyspark-03-types-functions-and-nulls.ipynb). Each exercise uses the `customers` and `orders` tables, or small inputs created in its data cell.

Write your answer in the empty cell under each exercise, run it, and compare your output with the expected output.
The reference answer is hidden under the exercise; open it when you are done or stuck.

*Last reviewed: November 2026, against Apache Spark 4.0.*

Run the input cell, try the exercise, then reveal the solution. The handbook site collapses tagged solutions. Jupyter and Colab use the saved collapse metadata when supported.

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

The first cell starts a local Spark session. The second creates `customers` (one row per customer; customer 4 has no region, and 7 and 8 have never ordered) and `orders` (one row per order; order 12 lands just after midnight UTC): a slice of the orders platform that every
module shares, introduced in [PySpark 1](pyspark-01-how-spark-runs.ipynb#setup-and-sample-data).

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

## Exercise 1: Type raw amounts and flag rejects

**Difficulty:** Easy · about 10 min

Convert `amount_text` to `decimal(12,2)` without failing on bad values. Return `order_id`, `amount` and
`is_rejected`, which is true when a value was delivered but could not be converted. Order by `order_id`.

**Sample input**

**`raw`**

| order_id | amount_text |
| --- | --- |
| o1 | 120.50 |
| o2 | N/A |
| o3 | null |
| o4 | 7.5 |

**Expected output**

| order_id | amount | is_rejected |
| --- | --- | --- |
| o1 | 120.50 | false |
| o2 | null | true |
| o3 | null | false |
| o4 | 7.50 | false |

In [ ]:
# Sample data for this exercise.
raw = spark.createDataFrame([("o1", "120.50"), ("o2", "N/A"), ("o3", None), ("o4", "7.5")], "order_id string, amount_text string")

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = (
    raw.withColumn("amount", F.expr("try_cast(amount_text AS DECIMAL(12,2))"))
    .withColumn("is_rejected", F.col("amount_text").isNotNull() & F.col("amount").isNull())
    .select("order_id", "amount", "is_rejected")
    .orderBy("order_id")
)
result.show()

## Exercise 2: Clean emails and extract the domain

**Difficulty:** Medium · about 10 min

Trim and lowercase each email. A null or blank email becomes `unknown`, and its domain is null. Return `id`,
`email_clean` and `domain`, ordered by `id`.

**Sample input**

**`contacts`**

| id | email |
| --- | --- |
| 1 |   Ada@Example.COM  |
| 2 |  |
| 3 | null |
| 4 | grace@Example.org |

**Expected output**

| id | email_clean | domain |
| --- | --- | --- |
| 1 | ada@example.com | example.com |
| 2 | unknown | null |
| 3 | unknown | null |
| 4 | grace@example.org | example.org |

In [ ]:
# Sample data for this exercise.
contacts = spark.createDataFrame([(1, "  Ada@Example.COM "), (2, ""), (3, None), (4, "grace@Example.org")], "id int, email string")

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
cleaned = contacts.withColumn("email_clean", F.coalesce(F.nullif(F.lower(F.trim("email")), F.lit("")), F.lit("unknown")))
result = (
    cleaned.withColumn(
        "domain", F.when(F.col("email_clean").contains("@"), F.element_at(F.split("email_clean", "@"), -1))
    )
    .select("id", "email_clean", "domain")
    .orderBy("id")
)
result.show()

## Exercise 3: Orders per business date in Chicago

**Difficulty:** Medium · about 10 min

Order timestamps are in UTC. Count orders per business date in `America/Chicago` time. Return `chicago_date` and
`orders`, ordered by date.

**Expected output**

| chicago_date | orders |
| --- | --- |
| 2026-01-05 | 3 |
| 2026-01-06 | 1 |
| 2026-01-07 | 2 |
| 2026-01-09 | 1 |
| 2026-01-10 | 1 |
| 2026-01-12 | 2 |
| 2026-01-13 | 2 |

In [ ]:
# Use the orders DataFrame.
orders.select("order_id", "order_ts").orderBy("order_id").show()

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = (
    orders.groupBy(F.to_date(F.from_utc_timestamp("order_ts", "America/Chicago")).alias("chicago_date"))
    .agg(F.count("*").alias("orders"))
    .orderBy("chicago_date")
)
result.show()

## Exercise 4: Size tiers

**Difficulty:** Easy · about 5 min

Assign each order a tier: `large` for 500 or more, `medium` for 100 or more, otherwise `small`. Return `order_id`,
`amount` and `tier`, ordered by `order_id`.

**Expected output**

| order_id | amount | tier |
| --- | --- | --- |
| 1 | 120.00 | medium |
| 2 | 84.50 | small |
| 3 | 30.00 | small |
| 4 | 260.00 | medium |
| 5 | 55.25 | small |
| 6 | 149.00 | medium |
| 7 | 310.00 | medium |
| 8 | 89.90 | small |
| 9 | 1180.00 | large |
| 10 | 74.00 | small |
| 11 | 210.00 | medium |
| 12 | 96.40 | small |

In [ ]:
# Use the orders DataFrame.
orders.select("order_id", "amount").orderBy("order_id").show()

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = orders.select(
    "order_id", "amount",
    F.when(F.col("amount") >= 500, "large").when(F.col("amount") >= 100, "medium").otherwise("small").alias("tier"),
).orderBy("order_id")
result.show()

## Exercise 5: Customers per region with a default

**Difficulty:** Medium · about 10 min

Count customers per region, reporting a missing region as `unknown`. Return `region` and `customers`, ordered by
`region`.

**Expected output**

| region | customers |
| --- | --- |
| East | 2 |
| North | 2 |
| South | 1 |
| West | 2 |
| unknown | 1 |

In [ ]:
# Use the customers DataFrame.
customers.select("customer_id", "region").orderBy("customer_id").show()

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = (
    customers.groupBy(F.coalesce("region", F.lit("unknown")).alias("region"))
    .agg(F.count("*").alias("customers"))
    .orderBy("region")
)
result.show()

## Exercise 6: Round money at the contract boundary

**Difficulty:** Medium · about 10 min

**Input.** `prices` has amount strings: ID 1 has 10.05 and ID 2 has null.

**Task.** Calculate amount after a 10% discount. Use decimal arithmetic and half-even rounding to cents. Keep unknown amounts null.

In [ ]:
prices = spark.createDataFrame([(1, "10.05"), (2, None)], "id int, amount string")

**Expected output**

| id | net |
| --- | --- |
| 1 | 9.04 |
| 2 | null |

In [ ]:
# Write your solution here.

In [ ]:
result = prices.select("id", F.bround(F.col("amount").cast("decimal(10,2)") * F.lit("0.90").cast("decimal(3,2)"), 2).alias("net")).orderBy("id")
result.show()

## Exercise 7: Build a replay-stable key

**Difficulty:** Medium · about 10 min

**Input.** `records` contains store/42 twice and web/42 once.

**Task.** Hash the structured source system and record ID. Identical input keys must hash alike, and the same ID in two systems must hash differently. Return counts instead of the hash strings.

In [ ]:
records = spark.createDataFrame([("store", 42), ("store", 42), ("web", 42)], "system string, id int")

**Expected output**

| rows | distinct_keys |
| --- | --- |
| 3 | 2 |

In [ ]:
# Write your solution here.

In [ ]:
keys = records.withColumn("key", F.sha2(F.to_json(F.struct("system", "id")), 256))
result = keys.agg(F.count("*").alias("rows"), F.countDistinct("key").alias("distinct_keys"))
result.show()

## Exercise 8: Map statuses to a lifecycle stage

**Difficulty:** Easy · about 5 min

Group each order's `status` into a stage: `placed` and `paid` are `open`, `shipped` is `fulfilled`, `returned` is
`returned`, and anything else is `unknown`. Use one `when` branch per stage, not one per status. Return `order_id`,
`status` and `stage`, ordered by `order_id`.

**Expected output**

| order_id | status | stage |
| --- | --- | --- |
| 1 | paid | open |
| 2 | shipped | fulfilled |
| 3 | returned | returned |
| 4 | paid | open |
| 5 | placed | open |
| 6 | shipped | fulfilled |
| 7 | paid | open |
| 8 | returned | returned |
| 9 | shipped | fulfilled |
| 10 | paid | open |
| 11 | placed | open |
| 12 | shipped | fulfilled |

In [ ]:
# Use the orders DataFrame.
orders.select("order_id", "status").orderBy("order_id").show()

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = orders.select(
    "order_id",
    "status",
    F.when(F.col("status").isin("placed", "paid"), "open")
    .when(F.col("status") == "shipped", "fulfilled")
    .when(F.col("status") == "returned", "returned")
    .otherwise("unknown")
    .alias("stage"),
).orderBy("order_id")
result.show()

## Exercise 9: Branch order decides the fee

**Difficulty:** Easy · about 10 min

Work out a `fee` for each shipment. The rules are listed in priority order: a returned shipment pays a 5.00
restocking fee whatever its amount; otherwise an amount of 150 or more ships free (0.00); everything else pays 9.99.
Return `id` and `fee` as `decimal(5,2)`, ordered by `id`. Check shipment 2: if the free-shipping rule runs first it
gets the wrong fee.

**Sample input**

**`shipments`**

| id | status | amount |
| --- | --- | --- |
| 1 | shipped | 40.00 |
| 2 | returned | 400.00 |
| 3 | shipped | 150.00 |
| 4 | returned | 20.00 |

**Expected output**

| id | fee |
| --- | --- |
| 1 | 9.99 |
| 2 | 5.00 |
| 3 | 0.00 |
| 4 | 5.00 |

In [ ]:
# Sample data for this exercise.
shipments = spark.createDataFrame(
    [(1, "shipped", D("40.00")), (2, "returned", D("400.00")), (3, "shipped", D("150.00")), (4, "returned", D("20.00"))],
    "id int, status string, amount decimal(10,2)",
)

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
fee = (
    F.when(F.col("status") == "returned", F.lit(D("5.00")))
    .when(F.col("amount") >= 150, F.lit(D("0.00")))
    .otherwise(F.lit(D("9.99")))
)
result = shipments.select("id", fee.cast("decimal(5,2)").alias("fee")).orderBy("id")
result.show()

## Exercise 10: A when with no otherwise

**Difficulty:** Easy · about 5 min

Add a `vip_note` that says `big spender` for orders of 1000 or more and is left empty (null) for every other order.
Do not write an `otherwise`. Then count how many orders got a note. Return `orders` and `with_note`.

**Expected output**

| orders | with_note |
| --- | --- |
| 12 | 1 |

In [ ]:
# Use the orders DataFrame.
orders.select("order_id", "amount").orderBy("order_id").show()

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
noted = orders.withColumn("vip_note", F.when(F.col("amount") >= 1000, "big spender"))
result = noted.agg(F.count("*").alias("orders"), F.count("vip_note").alias("with_note"))
result.show()

## Exercise 11: Three-valued logic in a flag

**Difficulty:** Medium · about 10 min

An order is a priority when it is express **and** its amount is 100 or more. `express` is sometimes unknown (null).
Return `id`, `raw_priority` (the plain boolean expression, nulls and all) and `priority` (the same flag with unknown
treated as false), ordered by `id`. Before you run it, predict `raw_priority` for orders 3 and 4.

**Sample input**

**`deliveries`**

| id | express | amount |
| --- | --- | --- |
| 1 | true | 150 |
| 2 | false | 150 |
| 3 | null | 150 |
| 4 | null | 50 |

**Expected output**

| id | raw_priority | priority |
| --- | --- | --- |
| 1 | true | true |
| 2 | false | false |
| 3 | null | false |
| 4 | false | false |

In [ ]:
# Sample data for this exercise.
deliveries = spark.createDataFrame([(1, True, 150), (2, False, 150), (3, None, 150), (4, None, 50)], "id int, express boolean, amount int")

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
raw = F.col("express") & (F.col("amount") >= 100)
result = deliveries.select("id", raw.alias("raw_priority"), F.coalesce(raw, F.lit(False)).alias("priority")).orderBy("id")
result.show()

## Exercise 12: Conditional aggregation per customer

**Difficulty:** Medium · about 10 min

For each customer with orders, return `orders` (all orders), `paid_orders` (orders with status `paid`) and
`net_amount`: the sum of amounts with returned orders subtracted instead of added. Order by `customer_id`.

**Expected output**

| customer_id | orders | paid_orders | net_amount |
| --- | --- | --- | --- |
| 1 | 3 | 1 | 174.50 |
| 2 | 2 | 1 | 315.25 |
| 3 | 2 | 1 | 459.00 |
| 4 | 2 | 0 | 1090.10 |
| 5 | 1 | 1 | 74.00 |
| 6 | 2 | 0 | 306.40 |

In [ ]:
# Use the orders DataFrame.
orders.select("order_id", "customer_id", "status", "amount").orderBy("order_id").show()

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = (
    orders.groupBy("customer_id")
    .agg(
        F.count("*").alias("orders"),
        F.count_if(F.col("status") == "paid").alias("paid_orders"),
        F.sum(F.when(F.col("status") == "returned", -F.col("amount")).otherwise(F.col("amount"))).alias("net_amount"),
    )
    .orderBy("customer_id")
)
result.show()

## Exercise 13: Replace a long when chain with a map

**Difficulty:** Medium · about 10 min

Status labels for the customer portal live in a small Python dict. Turn the dict into a map column with
`F.create_map` and look each order's status up in it, falling back to `Unknown status` when a status is missing from
the dict. Return `order_id` and `label`, ordered by `order_id`, for orders 1 to 5.

```python
labels = {"placed": "Order received", "paid": "Payment confirmed", "shipped": "On its way"}
```

**Expected output**

| order_id | label |
| --- | --- |
| 1 | Payment confirmed |
| 2 | On its way |
| 3 | Unknown status |
| 4 | Payment confirmed |
| 5 | Order received |

In [ ]:
# Sample data for this exercise.
labels = {"placed": "Order received", "paid": "Payment confirmed", "shipped": "On its way"}

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
from itertools import chain

label_map = F.create_map(*[F.lit(v) for v in chain.from_iterable(labels.items())])
result = (
    orders.where(F.col("order_id") <= 5)
    .select("order_id", F.coalesce(label_map[F.col("status")], F.lit("Unknown status")).alias("label"))
    .orderBy("order_id")
)
result.show()

## Exercise 14: List every data-quality problem in one column

**Difficulty:** Hard · about 15 min

Check each row against three rules: `email` is missing, `qty` is not positive, `country` is not one of `US`, `CA`,
`MX`. Return `id` and `issues`: every broken rule's name (`missing_email`, `bad_qty`, `bad_country`) joined with
`; `, or `ok` when the row breaks none. A null `qty` or `country` breaks its rule. Order by `id`.

**Sample input**

**`signups`**

| id | email | qty | country |
| --- | --- | --- | --- |
| 1 | a@x.com | 2 | US |
| 2 | null | 0 | US |
| 3 | c@x.com | null | BR |
| 4 | null | 5 | null |

**Expected output**

| id | issues |
| --- | --- |
| 1 | ok |
| 2 | missing_email; bad_qty |
| 3 | bad_qty; bad_country |
| 4 | missing_email; bad_country |

In [ ]:
# Sample data for this exercise.
signups = spark.createDataFrame(
    [(1, "a@x.com", 2, "US"), (2, None, 0, "US"), (3, "c@x.com", None, "BR"), (4, None, 5, None)],
    "id int, email string, qty int, country string",
)

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
checks = [
    F.when(F.col("email").isNull(), "missing_email"),
    F.when(~F.coalesce(F.col("qty") > 0, F.lit(False)), "bad_qty"),
    F.when(~F.coalesce(F.col("country").isin("US", "CA", "MX"), F.lit(False)), "bad_country"),
]
issues = F.concat_ws("; ", *checks)  # concat_ws skips the nulls a when-without-otherwise leaves
result = signups.select("id", F.when(issues == "", "ok").otherwise(issues).alias("issues")).orderBy("id")
result.show()

## Exercise 15: Customers outside the North region

**Difficulty:** Easy · about 5 min

Return every customer who is **not** in the North region, including the customer whose region is unknown. Return
`customer_id` and `region`, ordered by `customer_id`. A plain `region != 'North'` drops one row: find out which and
why.

**Expected output**

| customer_id | region |
| --- | --- |
| 3 | South |
| 4 | null |
| 5 | West |
| 6 | West |
| 7 | East |
| 8 | East |

In [ ]:
# Use the customers DataFrame.
customers.select("customer_id", "region").orderBy("customer_id").show()

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = (
    customers.where(~F.col("region").eqNullSafe("North"))
    .select("customer_id", "region")
    .orderBy("customer_id")
)
result.show()

## Exercise 16: Detect changed values between two snapshots

**Difficulty:** Medium · about 10 min

`snapshots` has each customer's region yesterday and today. Flag `changed` when the value differs, treating null
to null as unchanged and null to a value (or back) as changed. Return `customer_id` and `changed`, ordered by
`customer_id`.

**Sample input**

**`snapshots`**

| customer_id | region_old | region_new |
| --- | --- | --- |
| 1 | North | North |
| 2 | North | South |
| 3 | null | West |
| 4 | East | null |
| 5 | null | null |

**Expected output**

| customer_id | changed |
| --- | --- |
| 1 | false |
| 2 | true |
| 3 | true |
| 4 | true |
| 5 | false |

In [ ]:
# Sample data for this exercise.
snapshots = spark.createDataFrame(
    [(1, "North", "North"), (2, "North", "South"), (3, None, "West"), (4, "East", None), (5, None, None)],
    "customer_id int, region_old string, region_new string",
)

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = snapshots.select(
    "customer_id", (~F.col("region_old").eqNullSafe(F.col("region_new"))).alias("changed")
).orderBy("customer_id")
result.show()

## Exercise 17: Pick the first available phone and say where it came from

**Difficulty:** Medium · about 10 min

Choose one phone per contact: mobile first, then work, then home. Return `id`, `phone` (null if none) and `source`
(`mobile`, `work`, `home` or `none`), ordered by `id`. Treat a blank string as missing.

**Sample input**

**`phones`**

| id | mobile | work | home |
| --- | --- | --- | --- |
| 1 | 555-0101 | 555-0202 | null |
| 2 |  | 555-0303 | 555-0404 |
| 3 | null | null | 555-0505 |
| 4 | null | null | null |

**Expected output**

| id | phone | source |
| --- | --- | --- |
| 1 | 555-0101 | mobile |
| 2 | 555-0303 | work |
| 3 | 555-0505 | home |
| 4 | null | none |

In [ ]:
# Sample data for this exercise.
phones = spark.createDataFrame(
    [(1, "555-0101", "555-0202", None), (2, "", "555-0303", "555-0404"), (3, None, None, "555-0505"), (4, None, None, None)],
    "id int, mobile string, work string, home string",
)

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
cols = ["mobile", "work", "home"]
cleaned = phones.select("id", *[F.nullif(F.trim(c), F.lit("")).alias(c) for c in cols])
source = F.when(F.col("mobile").isNotNull(), "mobile").when(F.col("work").isNotNull(), "work")
source = source.when(F.col("home").isNotNull(), "home").otherwise("none")
result = cleaned.select("id", F.coalesce(*cols).alias("phone"), source.alias("source")).orderBy("id")
result.show()

## Exercise 18: Best and worst score across columns

**Difficulty:** Easy · about 5 min

Each student sat up to three quizzes; a missed quiz is null. Return `student`, `best` and `worst` across the three
columns, ignoring missed quizzes, ordered by `student`. Student `cy` missed every quiz.

**Sample input**

**`quizzes`**

| student | q1 | q2 | q3 |
| --- | --- | --- | --- |
| ana | 70 | 85 | 60 |
| ben | null | 90 | 75 |
| cy | null | null | null |

**Expected output**

| student | best | worst |
| --- | --- | --- |
| ana | 85 | 60 |
| ben | 90 | 75 |
| cy | null | null |

In [ ]:
# Sample data for this exercise.
quizzes = spark.createDataFrame([("ana", 70, 85, 60), ("ben", None, 90, 75), ("cy", None, None, None)], "student string, q1 int, q2 int, q3 int")

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = quizzes.select(
    "student", F.greatest("q1", "q2", "q3").alias("best"), F.least("q1", "q2", "q3").alias("worst")
).orderBy("student")
result.show()

## Exercise 19: Fill and drop nulls per column

**Difficulty:** Easy · about 10 min

Clean `carts`: drop rows where **both** `qty` and `channel` are null, then fill a missing `qty` with 0 and a missing
`channel` with `web`. Leave `coupon` nulls alone. Return all columns, ordered by `id`.

**Sample input**

**`carts`**

| id | qty | channel | coupon |
| --- | --- | --- | --- |
| 1 | 2 | app | null |
| 2 | null | null | SAVE10 |
| 3 | null | store | null |
| 4 | 1 | null | null |

**Expected output**

| id | qty | channel | coupon |
| --- | --- | --- | --- |
| 1 | 2 | app | null |
| 3 | 0 | store | null |
| 4 | 1 | web | null |

In [ ]:
# Sample data for this exercise.
carts = spark.createDataFrame(
    [(1, 2, "app", None), (2, None, None, "SAVE10"), (3, None, "store", None), (4, 1, None, None)],
    "id int, qty int, channel string, coupon string",
)

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = carts.na.drop(how="all", subset=["qty", "channel"]).na.fill({"qty": 0, "channel": "web"}).orderBy("id")
result.show()

## Exercise 20: Safe ratios without division errors

**Difficulty:** Medium · about 10 min

Return each store's `return_rate` (returns divided by orders, rounded to 2 places). A store with no orders, or an
unknown order count, gets null instead of an error. Spark 4 runs in ANSI mode, so a plain division by zero fails the
query. Return `store` and `return_rate`, ordered by `store`.

**Sample input**

**`store_stats`**

| store | returns | orders |
| --- | --- | --- |
| austin | 3 | 40 |
| boston | 0 | 0 |
| denver | 2 | null |
| eugene | 5 | 20 |

**Expected output**

| store | return_rate |
| --- | --- |
| austin | 0.08 |
| boston | null |
| denver | null |
| eugene | 0.25 |

In [ ]:
# Sample data for this exercise.
store_stats = spark.createDataFrame([("austin", 3, 40), ("boston", 0, 0), ("denver", 2, None), ("eugene", 5, 20)], "store string, returns int, orders int")

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = store_stats.select("store", F.round(F.try_divide("returns", "orders"), 2).alias("return_rate")).orderBy("store")
result.show()

## Exercise 21: Build full names when the middle name is missing

**Difficulty:** Easy · about 5 min

Return `id` and two versions of the full name: `with_concat` built with `concat` and `with_concat_ws` built with
`concat_ws(" ", ...)`, from first, middle and last name. Order by `id`. Note what happens to person 2.

**Sample input**

**`people`**

| id | first | middle | last |
| --- | --- | --- | --- |
| 1 | Ada | King | Lovelace |
| 2 | Alan | null | Turing |

**Expected output**

| id | with_concat | with_concat_ws |
| --- | --- | --- |
| 1 | Ada King Lovelace | Ada King Lovelace |
| 2 | null | Alan Turing |

In [ ]:
# Sample data for this exercise.
people = spark.createDataFrame([(1, "Ada", "King", "Lovelace"), (2, "Alan", None, "Turing")], "id int, first string, middle string, last string")

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = people.select(
    "id",
    F.concat("first", F.lit(" "), "middle", F.lit(" "), "last").alias("with_concat"),
    F.concat_ws(" ", "first", "middle", "last").alias("with_concat_ws"),
).orderBy("id")
result.show()

## Exercise 22: Parse codes with a regular expression

**Difficulty:** Medium · about 10 min

Reference codes look like `ORD-2026-00042`. Extract `year` and `seq` as integers. A code that does not match the
pattern gives null for both (careful: `regexp_extract` returns an empty string, not null, when nothing matches).
Return `ref`, `year` and `seq`, ordered by `ref`.

**Sample input**

**`refs`**

| ref |
| --- |
| ORD-2026-00042 |
| ORD-2025-01000 |
| ord-2026-7 |
| INV-2026-00001 |

**Expected output**

| ref | year | seq |
| --- | --- | --- |
| INV-2026-00001 | null | null |
| ORD-2025-01000 | 2025 | 1000 |
| ORD-2026-00042 | 2026 | 42 |
| ord-2026-7 | null | null |

In [ ]:
# Sample data for this exercise.
refs = spark.createDataFrame([("ORD-2026-00042",), ("ORD-2025-01000",), ("ord-2026-7",), ("INV-2026-00001",)], "ref string")

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
pattern = r"^ORD-(\d{4})-(\d{5})$"


def part(i):
    return F.nullif(F.regexp_extract("ref", pattern, i), F.lit("")).cast("int")


result = refs.select("ref", part(1).alias("year"), part(2).alias("seq")).orderBy("ref")
result.show()

## Exercise 23: Normalise phone numbers

**Difficulty:** Medium · about 10 min

Strip everything except digits from each phone number. If the result has 11 digits starting with `1`, drop the
leading `1`. Return `raw`, `digits` (the 10-digit number, or null when it is not 10 digits after that) and
`is_valid`, in input order (use `id`).

**Sample input**

**`raw_phones`**

| id | raw |
| --- | --- |
| 1 | (512) 555-0147 |
| 2 | +1 512.555.0188 |
| 3 | 555-0199 |
| 4 | null |

**Expected output**

| raw | digits | is_valid |
| --- | --- | --- |
| (512) 555-0147 | 5125550147 | true |
| +1 512.555.0188 | 5125550188 | true |
| 555-0199 | null | false |
| null | null | false |

In [ ]:
# Sample data for this exercise.
raw_phones = spark.createDataFrame([(1, "(512) 555-0147"), (2, "+1 512.555.0188"), (3, "555-0199"), (4, None)], "id int, raw string")

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
only_digits = F.regexp_replace("raw", r"\D", "")
ten = F.when((F.length(only_digits) == 11) & only_digits.startswith("1"), F.substring(only_digits, 2, 10)).otherwise(only_digits)
digits = F.when(F.length(ten) == 10, ten)
result = raw_phones.select("raw", digits.alias("digits"), digits.isNotNull().alias("is_valid")).orderBy("id")
result.show()

## Exercise 24: Format invoice numbers

**Difficulty:** Easy · about 5 min

Build an `invoice_no` for each order from its order date and ID: `INV-<yyyyMMdd>-<order_id padded to 4 digits>`,
for example `INV-20260105-0001`. Return `order_id` and `invoice_no` for orders 1, 9 and 12, ordered by `order_id`.

**Expected output**

| order_id | invoice_no |
| --- | --- |
| 1 | INV-20260105-0001 |
| 9 | INV-20260112-0009 |
| 12 | INV-20260114-0012 |

In [ ]:
# Use the orders DataFrame.
orders.select("order_id", "order_ts").orderBy("order_id").show()

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
invoice = F.concat_ws("-", F.lit("INV"), F.date_format("order_ts", "yyyyMMdd"), F.lpad(F.col("order_id").cast("string"), 4, "0"))
result = orders.where(F.col("order_id").isin(1, 9, 12)).select("order_id", invoice.alias("invoice_no")).orderBy("order_id")
result.show()

## Exercise 25: Split and tidy free-text addresses

**Difficulty:** Medium · about 10 min

`addresses.location` holds `city, state` typed by hand. Return `id`, `city` in title case with extra spaces removed,
and `state` as upper-case two letters. Order by `id`.

**Sample input**

**`addresses`**

| id | location |
| --- | --- |
| 1 | austin, tx |
| 2 |   NEW YORK ,ny |
| 3 | san  francisco, Ca |

**Expected output**

| id | city | state |
| --- | --- | --- |
| 1 | Austin | TX |
| 2 | New York | NY |
| 3 | San Francisco | CA |

In [ ]:
# Sample data for this exercise.
addresses = spark.createDataFrame([(1, "austin, tx"), (2, "  NEW YORK ,ny"), (3, "san  francisco, Ca")], "id int, location string")

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
city = F.initcap(F.regexp_replace(F.trim(F.substring_index("location", ",", 1)), r"\s+", " "))
state = F.upper(F.trim(F.substring_index("location", ",", -1)))
result = addresses.select("id", city.alias("city"), state.alias("state")).orderBy("id")
result.show()

## Exercise 26: Days from signup to first order

**Difficulty:** Medium · about 10 min

For every customer, return `customer_id`, `first_order_date` and `days_to_first_order` (days between `signup_date`
and the date of their first order). Customers who never ordered keep both as null. Order by `customer_id`.

**Expected output**

| customer_id | first_order_date | days_to_first_order |
| --- | --- | --- |
| 1 | 2026-01-05 | 733 |
| 2 | 2026-01-06 | 695 |
| 3 | 2026-01-07 | 663 |
| 4 | 2026-01-10 | 618 |
| 5 | 2026-01-12 | 571 |
| 6 | 2026-01-13 | 530 |
| 7 | null | null |
| 8 | null | null |

In [ ]:
# Use the customers and orders DataFrames.
customers.select("customer_id", "signup_date").orderBy("customer_id").show()
orders.select("order_id", "customer_id", "order_ts").orderBy("order_id").show()

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
first = orders.groupBy("customer_id").agg(F.to_date(F.min("order_ts")).alias("first_order_date"))
result = (
    customers.join(first, "customer_id", "left")
    .select("customer_id", "first_order_date", F.datediff("first_order_date", "signup_date").alias("days_to_first_order"))
    .orderBy("customer_id")
)
result.show()

## Exercise 27: Orders by day of the week

**Difficulty:** Medium · about 10 min

Count orders per weekday name (`Monday`, `Tuesday`, ...), sorted Monday first rather than alphabetically. Also flag
weekend days. Return `weekday`, `is_weekend` and `orders`. Spark's `dayofweek` numbers Sunday as 1.

**Expected output**

| weekday | is_weekend | orders |
| --- | --- | --- |
| Monday | false | 5 |
| Tuesday | false | 2 |
| Wednesday | false | 3 |
| Friday | false | 1 |
| Saturday | true | 1 |

In [ ]:
# Use the orders DataFrame.
orders.select("order_id", "order_ts").orderBy("order_id").show()

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
dow = F.dayofweek("order_ts")
result = (
    orders.groupBy(
        F.date_format("order_ts", "EEEE").alias("weekday"),
        dow.isin(1, 7).alias("is_weekend"),
        ((dow + 5) % 7).alias("monday_first"),
    )
    .agg(F.count("*").alias("orders"))
    .orderBy("monday_first")
    .drop("monday_first")
)
result.show()

## Exercise 28: Parse dates that arrive in two formats

**Difficulty:** Medium · about 10 min

`raw_dates.value` holds dates as `yyyy-MM-dd` or `MM/dd/yyyy`, plus some junk. Parse both formats into a `date`
column and return null for anything else without failing the job (ANSI mode makes `to_date` raise on bad input).
Return `id`, `value` and `parsed`, ordered by `id`.

**Sample input**

**`raw_dates`**

| id | value |
| --- | --- |
| 1 | 2026-01-05 |
| 2 | 01/31/2026 |
| 3 | 31/01/2026 |
| 4 | soon |

**Expected output**

| id | value | parsed |
| --- | --- | --- |
| 1 | 2026-01-05 | 2026-01-05 |
| 2 | 01/31/2026 | 2026-01-31 |
| 3 | 31/01/2026 | null |
| 4 | soon | null |

In [ ]:
# Sample data for this exercise.
raw_dates = spark.createDataFrame([(1, "2026-01-05"), (2, "01/31/2026"), (3, "31/01/2026"), (4, "soon")], "id int, value string")

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
parsed = F.coalesce(
    F.to_date(F.try_to_timestamp("value", F.lit("yyyy-MM-dd"))),
    F.to_date(F.try_to_timestamp("value", F.lit("MM/dd/yyyy"))),
)
result = raw_dates.select("id", "value", parsed.alias("parsed")).orderBy("id")
result.show()

## Exercise 29: Calendar arithmetic for renewals

**Difficulty:** Medium · about 10 min

For each customer who signed up in 2024 H2 (July onwards), return `customer_id`, `signup_date`, `signup_quarter`
(first day of the quarter), `month_end` (last day of the signup month) and `renewal_date` (12 months after signup).
Order by `customer_id`.

**Expected output**

| customer_id | signup_date | signup_quarter | month_end | renewal_date |
| --- | --- | --- | --- | --- |
| 6 | 2024-08-01 | 2024-07-01 | 2024-08-31 | 2025-08-01 |
| 7 | 2024-09-14 | 2024-07-01 | 2024-09-30 | 2025-09-14 |
| 8 | 2024-11-05 | 2024-10-01 | 2024-11-30 | 2025-11-05 |

In [ ]:
# Use the customers DataFrame.
customers.select("customer_id", "signup_date").orderBy("customer_id").show()

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = (
    customers.where(
        (F.col("signup_date") >= date(2024, 7, 1))
        & (F.col("signup_date") < date(2025, 1, 1))
    )
    .select(
        "customer_id",
        "signup_date",
        F.to_date(F.date_trunc("quarter", "signup_date")).alias("signup_quarter"),
        F.last_day("signup_date").alias("month_end"),
        F.add_months("signup_date", 12).alias("renewal_date"),
    )
    .orderBy("customer_id")
)
result.show()

## Exercise 30: Epoch milliseconds to timestamps and durations

**Difficulty:** Medium · about 10 min

A clickstream sends `start_ms` and `end_ms` as Unix epoch milliseconds. Return `session`, `started_at` as a
timestamp (UTC) and `minutes` (the session length in whole minutes, rounded down). Order by `session`.

**Sample input**

**`sessions`**

| session | start_ms | end_ms |
| --- | --- | --- |
| s1 | 1767603600000 | 1767605520000 |
| s2 | 1767690000000 | 1767690059000 |

**Expected output**

| session | started_at | minutes |
| --- | --- | --- |
| s1 | 2026-01-05 09:00:00 | 32 |
| s2 | 2026-01-06 09:00:00 | 0 |

In [ ]:
# Sample data for this exercise.
sessions = spark.createDataFrame([("s1", 1767603600000, 1767605520000), ("s2", 1767690000000, 1767690059000)], "session string, start_ms long, end_ms long")

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = sessions.select(
    "session",
    F.timestamp_millis("start_ms").alias("started_at"),
    F.floor((F.col("end_ms") - F.col("start_ms")) / 60000).alias("minutes"),
).orderBy("session")
result.show()

## Exercise 31: Bucket amounts into labelled bands

**Difficulty:** Easy · about 10 min

Put each order in a 100-wide band labelled like `0-99`, `100-199`, and so on (use `floor`). Count orders per band
and order the result by the band's lower bound. Return `band` and `orders`.

**Expected output**

| band | orders |
| --- | --- |
| 0-99 | 6 |
| 100-199 | 2 |
| 200-299 | 2 |
| 300-399 | 1 |
| 1100-1199 | 1 |

In [ ]:
# Use the orders DataFrame.
orders.select("order_id", "amount").orderBy("order_id").show()

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
low = (F.floor(F.col("amount") / 100) * 100).cast("int")
result = (
    orders.groupBy(low.alias("low"))
    .agg(F.count("*").alias("orders"))
    .orderBy("low")
    .select(F.concat_ws("-", F.col("low"), F.col("low") + 99).alias("band"), "orders")
)
result.show()

## Exercise 32: Cast a raw string table to proper types

**Difficulty:** Medium · about 15 min

Every column in `raw_items` arrived as a string. Return `sku` (string), `qty` (int), `price` (`decimal(8,2)`),
`active` (boolean) and `added` (date), ordered by `sku`. `active` uses `Y`/`N` in any case; anything else is null.
Bad numbers become null instead of failing the job.

**Sample input**

**`raw_items`**

| sku | qty | price | active | added |
| --- | --- | --- | --- | --- |
| A1 | 3 | 9.50 | Y | 2026-01-02 |
| B2 | three | 12 | n | 2026-01-03 |
| C3 | 7 | abc | maybe | 2026-01-04 |

**Expected output**

| sku | qty | price | active | added |
| --- | --- | --- | --- | --- |
| A1 | 3 | 9.50 | true | 2026-01-02 |
| B2 | null | 12.00 | false | 2026-01-03 |
| C3 | 7 | null | null | 2026-01-04 |

In [ ]:
# Sample data for this exercise.
raw_items = spark.createDataFrame(
    [("A1", "3", "9.50", "Y", "2026-01-02"), ("B2", "three", "12", "n", "2026-01-03"), ("C3", "7", "abc", "maybe", "2026-01-04")],
    "sku string, qty string, price string, active string, added string",
)

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
flag = F.upper(F.trim("active"))
result = raw_items.select(
    "sku",
    F.col("qty").try_cast("int").alias("qty"),
    F.col("price").try_cast("decimal(8,2)").alias("price"),
    F.when(flag == "Y", True).when(flag == "N", False).alias("active"),
    F.col("added").try_cast("date").alias("added"),
).orderBy("sku")
result.show()

## Exercise 33: Segment every customer

**Difficulty:** Hard · about 20 min

Label each customer with exactly one segment, checking the rules in this order:

1. `never ordered`: no orders at all
2. `high value`: net amount (returns subtracted) of 1000 or more
3. `returner`: at least one returned order
4. `regular`: everyone else

Return `customer_id`, `name` and `segment`, ordered by `customer_id`.

**Expected output**

| customer_id | name | segment |
| --- | --- | --- |
| 1 | Acme Corp | returner |
| 2 | Bright Retail | regular |
| 3 | Cyan Traders | regular |
| 4 | Delta Grocers | high value |
| 5 | Echo Supplies | regular |
| 6 | Foxtrot Foods | regular |
| 7 | Gamma Sports | never ordered |
| 8 | Hotel Homeware | never ordered |

In [ ]:
# Use the customers and orders DataFrames.
customers.select("customer_id", "name").orderBy("customer_id").show()
orders.select("order_id", "customer_id", "status", "amount").orderBy("order_id").show()

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
stats = orders.groupBy("customer_id").agg(
    F.sum(F.when(F.col("status") == "returned", -F.col("amount")).otherwise(F.col("amount"))).alias("net"),
    F.count("*").alias("order_count"),
    F.count_if(F.col("status") == "returned").alias("returns"),
)
segment = (
    F.when(F.coalesce(F.col("order_count"), F.lit(0)) == 0, "never ordered")
    .when(F.col("net") >= 1000, "high value")
    .when(F.col("returns") > 0, "returner")
    .otherwise("regular")
)
result = (
    customers.join(stats, "customer_id", "left")
    .select("customer_id", "name", segment.alias("segment"))
    .orderBy("customer_id")
)
result.show()

**Next:** [PySpark 4: reading and writing data](pyspark-04-reading-and-writing-data.ipynb)